# 🧭 Bússola Pública — Etapa 1: Padronização (Bronze → Silver)

**Responsável:** Integrante 2 — Engenharia de Dados

Lê os retornos brutos salvos pelo `01_extracao_api_camara.ipynb` em `data/raw/` e entrega DataFrames **limpos e no formato do Star Schema** do README.

| Regra | O que faz |
| :--- | :--- |
| **Nomes** | `camelCase` da API → `snake_case` com os nomes das colunas do banco (`id` → `id_proposicao`, `siglaTipo` → `sigla_tipo`...) |
| **Tipos** | datas viram `datetime`, ids viram inteiros, `aprovacao` vira booleano |
| **Textos** | `strip`, espaços/quebras de linha repetidos viram um espaço só, texto vazio vira nulo |
| **Deduplicação** | uma linha por chave primária |
| **Validação** | `assert` de chave única e campos obrigatórios antes de salvar |

> ⚠️ Rode o notebook 01 antes, para existir `data/raw/*.json`.

## 1. Imports e leitura da camada Bronze

In [1]:
import hashlib
import json
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PASTA_RAW = Path("../data/raw")
PASTA_SILVER = Path("../data/silver")
ATUALIZADO_EM = pd.Timestamp(datetime.now()).floor("s")


def ler_bronze(nome: str) -> pd.DataFrame:
    caminho = PASTA_RAW / f"{nome}.json"
    df = pd.DataFrame(json.loads(caminho.read_text(encoding="utf-8")))
    print(f"📥 {caminho}: {df.shape[0]} linhas × {df.shape[1]} colunas")
    return df


bronze = {nome: ler_bronze(nome) for nome in ["partidos", "deputados", "proposicoes", "votacoes", "despesas"]}

📥 ..\data\raw\partidos.json: 21 linhas × 4 colunas
📥 ..\data\raw\deputados.json: 513 linhas × 9 colunas
📥 ..\data\raw\proposicoes.json: 7514 linhas × 8 colunas
📥 ..\data\raw\votacoes.json: 2004 linhas × 11 colunas
📥 ..\data\raw\despesas.json: 3627 linhas × 18 colunas


## 2. Funções de padronização
Funções reaproveitadas por todas as tabelas.

In [2]:
def limpar_texto(serie: pd.Series) -> pd.Series:
    """strip + colapsa espaços/quebras de linha + texto vazio vira nulo."""
    return (
        serie.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


def padronizar(df: pd.DataFrame, colunas: dict[str, str]) -> pd.DataFrame:
    """Seleciona e renomeia colunas (API → banco), na ordem do dicionário."""
    return df[list(colunas)].rename(columns=colunas).copy()


def deduplicar(df: pd.DataFrame, chave: str | list[str], nome: str) -> pd.DataFrame:
    antes = len(df)
    df = df.drop_duplicates(subset=chave, keep="last").reset_index(drop=True)
    print(f"🧹 {nome}: {antes - len(df)} duplicado(s) removido(s) pela chave {chave}")
    return df


def validar(df: pd.DataFrame, pk: str, obrigatorias: list[str], nome: str) -> None:
    assert df[pk].is_unique, f"{nome}: chave {pk} repetida"
    for col in [pk, *obrigatorias]:
        nulos = df[col].isna().sum()
        assert nulos == 0, f"{nome}: {nulos} nulo(s) na coluna obrigatória {col}"
    print(f"✅ {nome}: {len(df)} linhas válidas — PK `{pk}` única, obrigatórias sem nulos")

## 3. `dim_partidos`

In [3]:
dim_partidos = padronizar(bronze["partidos"], {
    "id": "id_partido",
    "sigla": "sigla",
    "nome": "nome",
})
dim_partidos["id_partido"] = dim_partidos["id_partido"].astype("Int64")
dim_partidos["sigla"] = limpar_texto(dim_partidos["sigla"]).str.upper()
dim_partidos["nome"] = limpar_texto(dim_partidos["nome"])
dim_partidos["atualizado_em"] = ATUALIZADO_EM

dim_partidos = deduplicar(dim_partidos, "id_partido", "dim_partidos")
validar(dim_partidos, "id_partido", ["sigla", "nome"], "dim_partidos")
dim_partidos.head()

🧹 dim_partidos: 0 duplicado(s) removido(s) pela chave id_partido
✅ dim_partidos: 21 linhas válidas — PK `id_partido` única, obrigatórias sem nulos


,id_partido,sigla,nome,atualizado_em
0,36898,AVANTE,Avante,2026-11-26 20:58:40
1,37905,CIDADANIA,Cidadania,2026-11-26 20:58:40
2,36899,MDB,Movimento Democrático Brasileiro,2026-11-26 20:58:40
3,38011,MISSÃO,Partido Missão,2026-11-26 20:58:40
4,37901,NOVO,Partido Novo,2026-11-26 20:58:40


> ℹ️ A coluna `status` do schema não vem na listagem `/partidos` (só em `/partidos/{id}`). Fica para decidir na Etapa 2 se vale a chamada extra.

## 4. `dim_deputados`

In [4]:
dim_deputados = padronizar(bronze["deputados"], {
    "id": "id_deputado",
    "nome": "nome",
    "siglaPartido": "sigla_partido",
    "siglaUf": "sigla_uf",
    "urlFoto": "url_foto",
    "email": "email",
})
dim_deputados["id_deputado"] = dim_deputados["id_deputado"].astype("Int64")
for col in ["nome", "url_foto"]:
    dim_deputados[col] = limpar_texto(dim_deputados[col])
for col in ["sigla_partido", "sigla_uf"]:
    dim_deputados[col] = limpar_texto(dim_deputados[col]).str.upper()
dim_deputados["email"] = limpar_texto(dim_deputados["email"]).str.lower()
dim_deputados["atualizado_em"] = ATUALIZADO_EM

dim_deputados = deduplicar(dim_deputados, "id_deputado", "dim_deputados")
validar(dim_deputados, "id_deputado", ["nome"], "dim_deputados")

# Integridade: todo partido dos deputados existe na dim_partidos?
sem_partido = set(dim_deputados["sigla_partido"].dropna()) - set(dim_partidos["sigla"])
print("Siglas de partido sem correspondência:", sem_partido or "nenhuma")
dim_deputados.head()

🧹 dim_deputados: 0 duplicado(s) removido(s) pela chave id_deputado
✅ dim_deputados: 513 linhas válidas — PK `id_deputado` única, obrigatórias sem nulos
Siglas de partido sem correspondência: nenhuma


,id_deputado,nome,sigla_partido,sigla_uf,url_foto,email,atualizado_em
0,204379,Acácio Favacho,MDB,AP,https://www.camara.leg.br/internet/deputado/bandep/204379.jpg,dep.acaciofavacho@camara.leg.br,2026-11-26 20:58:40
1,220714,Adail Filho,MDB,AM,https://www.camara.leg.br/internet/deputado/bandep/220714.jpg,dep.adailfilho@camara.leg.br,2026-11-26 20:58:40
2,221328,Adilson Barroso,PL,SP,https://www.camara.leg.br/internet/deputado/bandep/221328.jpg,dep.adilsonbarroso@camara.leg.br,2026-11-26 20:58:40
3,204560,Adolfo Viana,PSDB,BA,https://www.camara.leg.br/internet/deputado/bandep/204560.jpg,dep.adolfoviana@camara.leg.br,2026-11-26 20:58:40
4,204528,Adriana Ventura,NOVO,SP,https://www.camara.leg.br/internet/deputado/bandep/204528.jpg,dep.adrianaventura@camara.leg.br,2026-11-26 20:58:40


## 5. `fato_proposicoes`
- `dataApresentacao` (`"2026-10-27T10:13"`) vira `datetime`.
- Ementas têm quebras de linha e espaços duplos: são normalizadas.
- Ementas vazias viram nulo (não dá para classificar com IA sem texto).

In [5]:
fato_proposicoes = padronizar(bronze["proposicoes"], {
    "id": "id_proposicao",
    "siglaTipo": "sigla_tipo",
    "numero": "numero",
    "ano": "ano",
    "ementa": "ementa",
    "dataApresentacao": "data_apresentacao",
    "uri": "uri",
})
for col in ["id_proposicao", "numero", "ano"]:
    fato_proposicoes[col] = fato_proposicoes[col].astype("Int64")
fato_proposicoes["sigla_tipo"] = limpar_texto(fato_proposicoes["sigla_tipo"]).str.upper()
fato_proposicoes["ementa"] = limpar_texto(fato_proposicoes["ementa"])
fato_proposicoes["uri"] = limpar_texto(fato_proposicoes["uri"])
fato_proposicoes["data_apresentacao"] = pd.to_datetime(fato_proposicoes["data_apresentacao"], errors="coerce")

fato_proposicoes = deduplicar(fato_proposicoes, "id_proposicao", "fato_proposicoes")
validar(fato_proposicoes, "id_proposicao", ["sigla_tipo", "numero", "ano", "data_apresentacao"], "fato_proposicoes")

print(f"Ementas vazias (viraram nulo): {fato_proposicoes['ementa'].isna().sum()}")
fato_proposicoes.head()

🧹 fato_proposicoes: 0 duplicado(s) removido(s) pela chave id_proposicao
✅ fato_proposicoes: 7514 linhas válidas — PK `id_proposicao` única, obrigatórias sem nulos
Ementas vazias (viraram nulo): 84


,id_proposicao,sigla_tipo,numero,ano,ementa,data_apresentacao,uri
0,2648935,PRL,1,0,"Parecer do Relator, Dep. Diego Garcia (UNIÃO-PR), pela aprovação, com emenda.",2026-10-29 09:07:00,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648935
1,2648936,EMR,1,0,Reconhece como manifestação da cultura nacional o Festejo do Santuário Dioce...,2026-10-29 09:07:00,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648936
2,2648938,PRL,1,0,"Parecer do Relator, Dep. Cleber Verde (MDB-MA), pela incompatibilidade e ina...",2026-10-27 10:13:00,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648938
3,2648939,PL,5663,2026,Dispõe sobre medidas de proteção à mulher em situação de violência doméstica...,2026-10-27 10:23:00,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648939
4,2648940,PL,5664,2026,"Altera a Lei nº 9.797, de 6 de maio de 1999, para instituir o Kit de Reabili...",2026-10-27 10:23:00,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648940


### Tipos de proposição
A maioria **não** são projetos de lei: `RPD`, `PRL`, `PAR`, `REQ`... são documentos acessórios (pareceres, requerimentos, redações).
A extração traz tudo; a decisão de filtrar (ex.: só `PL`, `PLP`, `PEC`, `MPV`, `PDL`) fica para o squad, já que afeta a camada de IA.

In [6]:
fato_proposicoes["sigla_tipo"].value_counts().head(15)

sigla_tipo
RPD      1734
PRL      1143
PAR       842
REQ       514
SBT       300
PL        295
SLD       295
PDL       279
RIC       267
DOC       261
EMC       211
SBT-A     171
RDF       142
PARF      127
EMP       104
Name: count, dtype: Int64

## 6. `fato_votacoes`
- `id` da votação é texto (`"949156-136"`), então `id_votacao` continua `string`.
- `id_proposicao` (FK) é extraído do final de `uriProposicaoObjeto`. Muitas votações não têm proposição associada (requerimentos de sessão etc.), e aí fica nulo.
- `aprovacao` vem como `1.0 / 0.0 / NaN` e vira booleano que aceita nulo.

In [7]:
vot = bronze["votacoes"].copy()
vot["id_proposicao"] = (
    vot["uriProposicaoObjeto"].astype("string").str.extract(r"/proposicoes/(\d+)$", expand=False).astype("Int64")
)

fato_votacoes = padronizar(vot, {
    "id": "id_votacao",
    "id_proposicao": "id_proposicao",
    "data": "data_votacao",
    "descricao": "descricao",
    "aprovacao": "aprovacao",
})
fato_votacoes["id_votacao"] = limpar_texto(fato_votacoes["id_votacao"])
fato_votacoes["data_votacao"] = pd.to_datetime(fato_votacoes["data_votacao"], errors="coerce").dt.date
fato_votacoes["descricao"] = limpar_texto(fato_votacoes["descricao"])
fato_votacoes["aprovacao"] = fato_votacoes["aprovacao"].map({1.0: True, 0.0: False}).astype("boolean")

fato_votacoes = deduplicar(fato_votacoes, "id_votacao", "fato_votacoes")
validar(fato_votacoes, "id_votacao", ["data_votacao"], "fato_votacoes")

print(f"Votações com proposição associada: {fato_votacoes['id_proposicao'].notna().sum()} de {len(fato_votacoes)}")
print("aprovacao:", fato_votacoes["aprovacao"].value_counts(dropna=False).to_dict())
fato_votacoes.head()

🧹 fato_votacoes: 0 duplicado(s) removido(s) pela chave id_votacao
✅ fato_votacoes: 2004 linhas válidas — PK `id_votacao` única, obrigatórias sem nulos
Votações com proposição associada: 688 de 2004
aprovacao: {np.True_: 1964, np.False_: 35, <NA>: 5}


,id_votacao,id_proposicao,data_votacao,descricao,aprovacao
0,2532113-40,<NA>,2026-11-26,Aprovada a Redação Final assinada pelo Dep. Afonso Florence (PT-BA).,True
1,2532113-37,<NA>,2026-11-26,"Aprovada a Subemenda Substitutiva ao Projeto de Decreto Legislativo nº 366, ...",True
2,2516578-35,<NA>,2026-11-26,Aprovada a Redação Final assinada pelo Dep. Ricardo Ayres (Republicanos/TO).,True
3,2516578-33,<NA>,2026-11-26,"Aprovado o Projeto de Decreto Legislativo nº 235, de 2025.",True
4,949156-136,<NA>,2026-11-25,"Aprovada a Redação Final assinada pelo relator, Dep. Thiago de Joaldo (REPUB...",True


## 7. `fato_despesas`
⚠️ **`codDocumento` não é único** (o schema do README prevê `cod_documento UNIQUE`):
- **Telefonia** vem com `codDocumento = "0"` em vários lançamentos (estornos e ressarcimentos com valor negativo).
- **Passagens aéreas (SIGEPA)** repetem o mesmo código (GUID) em várias linhas, uma por trecho ou bilhete.

Se usássemos `cod_documento` como chave, perderíamos lançamentos reais. A solução é a chave **`id_despesa` = hash MD5 determinístico** do lançamento completo: ela é a mesma em toda reexecução, o que serve para o `UPSERT` da Etapa 2.

Outras regras:
- `cnpj_cpf_fornecedor`: só dígitos (14 = CNPJ, 11 = CPF).
- `tipo_despesa`: remove o ponto final que a API às vezes coloca (`"COMBUSTÍVEIS E LUBRIFICANTES."`).

In [8]:
desp = bronze["despesas"].copy()
print("codDocumento repetidos na Bronze:", desp["codDocumento"].duplicated().sum())

CAMPOS_HASH = [
    "idDeputado", "ano", "mes", "tipoDespesa", "codDocumento", "numDocumento",
    "dataDocumento", "valorDocumento", "valorLiquido", "valorGlosa",
    "cnpjCpfFornecedor", "numRessarcimento", "codLote", "parcela",
]


def gerar_id_despesa(linha: pd.Series) -> str:
    chave = "|".join("" if pd.isna(linha[c]) else str(linha[c]) for c in CAMPOS_HASH)
    return hashlib.md5(chave.encode("utf-8")).hexdigest()


desp["id_despesa"] = desp.apply(gerar_id_despesa, axis=1)

fato_despesas = padronizar(desp, {
    "id_despesa": "id_despesa",
    "codDocumento": "cod_documento",
    "idDeputado": "id_deputado",
    "ano": "ano",
    "mes": "mes",
    "tipoDespesa": "tipo_despesa",
    "valorDocumento": "valor_documento",
    "valorGlosa": "valor_glosa",
    "valorLiquido": "valor_liquido",
    "dataDocumento": "data_documento",
    "nomeFornecedor": "nome_fornecedor",
    "cnpjCpfFornecedor": "cnpj_cpf_fornecedor",
    "urlDocumento": "url_documento",
})
for col in ["id_deputado", "ano", "mes"]:
    fato_despesas[col] = fato_despesas[col].astype("Int64")
for col in ["valor_documento", "valor_glosa", "valor_liquido"]:
    fato_despesas[col] = pd.to_numeric(fato_despesas[col], errors="coerce").round(2)
fato_despesas["valor_glosa"] = fato_despesas["valor_glosa"].fillna(0.0)
fato_despesas["data_documento"] = pd.to_datetime(fato_despesas["data_documento"], errors="coerce").dt.date
fato_despesas["cod_documento"] = limpar_texto(fato_despesas["cod_documento"])
fato_despesas["tipo_despesa"] = limpar_texto(fato_despesas["tipo_despesa"]).str.upper().str.rstrip(".")
fato_despesas["nome_fornecedor"] = limpar_texto(fato_despesas["nome_fornecedor"])
fato_despesas["cnpj_cpf_fornecedor"] = (
    limpar_texto(fato_despesas["cnpj_cpf_fornecedor"]).str.replace(r"\D", "", regex=True).replace("", pd.NA)
)
fato_despesas["url_documento"] = limpar_texto(fato_despesas["url_documento"])

fato_despesas = deduplicar(fato_despesas, "id_despesa", "fato_despesas")
validar(fato_despesas, "id_despesa", ["id_deputado", "ano", "mes", "tipo_despesa", "valor_liquido"], "fato_despesas")

# Integridade: todo deputado das despesas existe na dim_deputados?
orfas = ~fato_despesas["id_deputado"].isin(dim_deputados["id_deputado"])
print(f"Despesas sem deputado na dimensão: {orfas.sum()}")
print("Tamanho do documento do fornecedor:", fato_despesas["cnpj_cpf_fornecedor"].str.len().value_counts().to_dict())
fato_despesas.head()

codDocumento repetidos na Bronze: 193


🧹 fato_despesas: 0 duplicado(s) removido(s) pela chave id_despesa
✅ fato_despesas: 3627 linhas válidas — PK `id_despesa` única, obrigatórias sem nulos
Despesas sem deputado na dimensão: 0
Tamanho do documento do fornecedor: {np.int64(14): 3602, np.int64(11): 25}


,id_despesa,cod_documento,id_deputado,ano,mes,tipo_despesa,valor_documento,valor_glosa,valor_liquido,data_documento,nome_fornecedor,cnpj_cpf_fornecedor,url_documento
0,8e98452c1f775cc6e710c3e173910077,8046356,204379,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR,5000.00,0.0,-5000.00,2026-01-29,VN & FW LTDA-M,22209187000170,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046356...
1,6d010bb0bff3668871db37d785226fc7,8046357,204379,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR,6200.00,0.0,-6200.00,2026-01-27,QUINTELA & SA SERVIÇOS LTDA,31277848000196,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046357...
2,dd96a627cfd081dc99f2fd889b6e694c,8046358,204379,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR,1500.00,0.0,-1500.00,2026-02-02,Eka Comunicacao,58744132000108,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046358...
3,922f91968064f31f554f34017fbb77fb,8054458,204379,2026,1,COMBUSTÍVEIS E LUBRIFICANTES,341.99,0.0,-341.99,2026-01-29,POSTO IPANEMA TEREZOPOLIS I LTDA,62411109000179,http://www.camara.leg.br/cota-parlamentar/nota-fiscal-eletronica?ideDocument...
4,72ae06bb7521f51d9cb4239ea27d6c34,8054468,204379,2026,1,COMBUSTÍVEIS E LUBRIFICANTES,303.65,0.0,-303.65,2026-01-15,POSTO IPANEMA TEREZOPOLIS I LTDA,62411109000179,http://www.camara.leg.br/cota-parlamentar/nota-fiscal-eletronica?ideDocument...


## 8. Relatório de qualidade (Silver)

In [9]:
silver = {
    "dim_partidos": dim_partidos,
    "dim_deputados": dim_deputados,
    "fato_proposicoes": fato_proposicoes,
    "fato_votacoes": fato_votacoes,
    "fato_despesas": fato_despesas,
}

relatorio = pd.DataFrame([
    {
        "tabela": nome,
        "linhas_bronze": len(bronze[b]),
        "linhas_silver": len(df),
        "colunas": df.shape[1],
        "colunas_com_nulo": ", ".join(f"{c} ({n})" for c, n in df.isna().sum().items() if n) or "—",
    }
    for (nome, df), b in zip(silver.items(), ["partidos", "deputados", "proposicoes", "votacoes", "despesas"])
])
relatorio

,tabela,linhas_bronze,linhas_silver,colunas,colunas_com_nulo
0,dim_partidos,21,21,4,—
1,dim_deputados,513,513,7,—
2,fato_proposicoes,7514,7514,7,ementa (84)
3,fato_votacoes,2004,2004,5,"id_proposicao (1316), aprovacao (5)"
4,fato_despesas,3627,3627,13,"data_documento (88), url_documento (908)"


In [10]:
# Tipos finais de cada tabela
pd.concat({nome: df.dtypes.astype(str) for nome, df in silver.items()}, axis=0).to_frame("dtype")

dtype
dim_partidos     id_partido                    Int64
                 sigla                        string
                 nome                         string
                 atualizado_em        datetime64[us]
dim_deputados    id_deputado                   Int64
                 nome                         string
                 sigla_partido                string
                 sigla_uf                     string
                 url_foto                     string
                 email                        string
                 atualizado_em        datetime64[us]
fato_proposicoes id_proposicao                 Int64
                 sigla_tipo                   string
                 numero                        Int64
                 ano                           Int64
                 ementa                       string
                 data_apresentacao    datetime64[us]
                 uri                          string
fato_votacoes    id_votacao                   string
                 id_proposicao                 Int64
                 data_votacao                 object
                 descricao                    string
                 aprovacao                   boolean
fato_despesas    id_despesa                      str
                 cod_documento                string
                 id_deputado                   Int64
                 ano                           Int64
                 mes                           Int64
                 tipo_despesa                 string
                 valor_documento             float64
                 valor_glosa                 float64
                 valor_liquido               float64
                 data_documento               object
                 nome_fornecedor              string
                 cnpj_cpf_fornecedor          string
                 url_documento                string

## 9. Salvar a camada Silver (`data/silver/`)
JSON por registro com datas em ISO 8601 e nulos como `null`. `data/silver/` está no `.gitignore`.

In [11]:
PASTA_SILVER.mkdir(parents=True, exist_ok=True)
for nome, df in silver.items():
    caminho = PASTA_SILVER / f"{nome}.json"
    df.to_json(caminho, orient="records", date_format="iso", force_ascii=False, indent=0)
    print(f"💾 {caminho} — {len(df)} linhas")

💾 ..\data\silver\dim_partidos.json — 21 linhas
💾 ..\data\silver\dim_deputados.json — 513 linhas


💾 ..\data\silver\fato_proposicoes.json — 7514 linhas
💾 ..\data\silver\fato_votacoes.json — 2004 linhas


💾 ..\data\silver\fato_despesas.json — 3627 linhas


> **Próximo passo:** levar as funções dos notebooks 01 e 02 para `src/etl_camara.py`, que é o entregável da Etapa 1, e na Etapa 2 fazer o `UPSERT` no Supabase.